In [1]:
import sys, os

sys.path.insert(0, os.path.abspath("C:/Users/Voror/Projects/Personal/sep"))

from sklearn.model_selection import KFold
from SIDER_dataset.libraries.XofN_library import *
from SIDER_dataset.libraries.PCT_library import run_PCT, save_to_arff
from SIDER_dataset.libraries.utils import *
import json
from collections import Counter
import re

%load_ext autoreload
%autoreload 2

In [2]:
# Set ADR to predict and scoring
clus_path = get_clus_path()
paths = get_dataset_paths()
print(len(paths), "datasets")
paths

18 datasets


[{'dataset_path': 'C:\\Users\\Voror\\Projects\\Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_cardiac.csv',
  'dataset_name': 'CPI+fingerprint_cardiac',
  'label_set': ['se_C0016382',
   'se_C0018799',
   'se_C0003811',
   'se_C0428977',
   'se_C0027051',
   'se_C0018790'],
  'features': 2147,
  'original_features': ['cpi_9606.ENSP00000000442',
   'cpi_9606.ENSP00000001008',
   'cpi_9606.ENSP00000003084',
   'cpi_9606.ENSP00000003100',
   'cpi_9606.ENSP00000005178',
   'cpi_9606.ENSP00000011292',
   'cpi_9606.ENSP00000011653',
   'cpi_9606.ENSP00000012443',
   'cpi_9606.ENSP00000013034',
   'cpi_9606.ENSP00000014930',
   'cpi_9606.ENSP00000019103',
   'cpi_9606.ENSP00000023897',
   'cpi_9606.ENSP00000039007',
   'cpi_9606.ENSP00000044462',
   'cpi_9606.ENSP00000054668',
   'cpi_9606.ENSP00000078429',
   'cpi_9606.ENSP00000155840',
   'cpi_9606.ENSP00000164139',
   'cpi_9606.ENSP00000171757',
   'cpi_9606.ENSP00000176183',
   'cpi_9606.ENSP00000176195',
 

In [3]:
paths = [path for path in paths if "ten_mid" in path["dataset_name"]]
len(paths)

3

In [3]:
# # ONLY ONCE - DONE
# k = 10
# random_state = 42
# vouk_path = "C:/Users/Voror/Projects/Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/vouk_folds"
#
# for idx, path in enumerate(paths, start=1):
#     features = path["original_features"]
#     for label in path['label_set']:
#         run_config = f"Running with dataset:'{path["dataset_name"]}' and label:'{label}'"
#         print(run_config)
#         current_df = pd.read_csv(path["dataset_path"])
#         current_df = current_df[features + [label]].copy()
#         kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
#         for fold, (train_idx, test_idx) in enumerate(kf.split(current_df), start=1):
#             print(f"\nFold {fold}/{k} ({path["dataset_name"]} {idx}/{len(paths)})")
#             train_dataset = current_df.iloc[train_idx]
#             test_dataset = current_df.iloc[test_idx]
#             train_path = f"{vouk_path}/{path['dataset_name']}_{label}_trainFold_{fold}.csv"
#             test_path = f"{vouk_path}/{path['dataset_name']}_{label}_testFold_{fold}.csv"
#             print(train_path, test_path)
#             train_dataset.to_csv(train_path, index=False)
#             test_dataset.to_csv(test_path, index=False)
#             save_to_arff(train_path, [label])
#             save_to_arff(test_path, [label])
# # 16m for 18 datasets

Running with dataset:'CPI+fingerprint_cardiac' and label:'se_C0016382'

Fold 1/10 (CPI+fingerprint_cardiac 1/18)
C:/Users/Voror/Projects/Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/vouk_folds/CPI+fingerprint_cardiac_se_C0016382_trainFold_1.csv C:/Users/Voror/Projects/Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/vouk_folds/CPI+fingerprint_cardiac_se_C0016382_testFold_1.csv

Fold 2/10 (CPI+fingerprint_cardiac 1/18)
C:/Users/Voror/Projects/Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/vouk_folds/CPI+fingerprint_cardiac_se_C0016382_trainFold_2.csv C:/Users/Voror/Projects/Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/vouk_folds/CPI+fingerprint_cardiac_se_C0016382_testFold_2.csv

Fold 3/10 (CPI+fingerprint_cardiac 1/18)
C:/Users/Voror/Projects/Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/vouk_folds/CPI+fingerprint_cardiac_se_C0016382_trainFold_3.csv C:/Users/Voror/Projects/Personal/sep/SIDER_dataset/data

In [4]:
def print_candidate_features_stats(candidate_features_for_all_labels: dict):
    for label, groups in candidate_features_for_all_labels.items():
        num_groups = len(groups)

        if num_groups == 0:
            print(f"Label: {label}, Number of groups: 0")
            continue

        lengths = [len(group) for group in groups]
        counts = Counter(lengths)

        avg_len = sum(lengths) / num_groups
        min_len = min(lengths)
        max_len = max(lengths)

        print(f"\nLabel: {label}")
        print(f"Number of groups: {num_groups}")
        print("Group length distribution:")

        for length in sorted(counts):
            print(f"  Length {length}: {counts[length]} groups")

        print(
            f"Avg length: {avg_len:.2f}, "
            f"Min length: {min_len}, "
            f"Max length: {max_len}"
        )
        return avg_len


def apply_max_size_to_full_list(candidate_features_for_all_labels, max_size):
    return {
        label: [group[:max_size] for group in groups]
        for label, groups in candidate_features_for_all_labels.items()
    }


def deduplicate_full_list_groups(candidate_features_capped):
    return {
        label: list({tuple(g): g for g in groups}.values())
        for label, groups in candidate_features_capped.items()
    }


def get_full_list_intersection(candidate_features_unique):
    first_label_groups = next(iter(candidate_features_unique.values()))
    other_sets = [
        set(tuple(group) for group in groups)
        for groups in list(candidate_features_unique.values())[1:]
    ]

    if not other_sets:
        return first_label_groups.copy()

    common = set.intersection(*other_sets)
    return {"intersection": [g for g in first_label_groups if tuple(g) in common]}


def get_full_list_union(candidate_features_unique):
    union = set()
    for groups in candidate_features_unique.values():
        for group in groups:
            union.add(tuple(group))  # convert to tuple for hashing
    return {"union": [list(group) for group in union]}


def get_full_list_vouk(candidate_features_for_all_labels, max_size):
    # print_candidate_features_stats(candidate_features_for_all_labels)
    candidate_features_capped = apply_max_size_to_full_list(candidate_features_for_all_labels, max_size)
    candidate_features_unique = deduplicate_full_list_groups(candidate_features_capped)
    candidate_features_union = get_full_list_union(candidate_features_unique)
    avg_features = print_candidate_features_stats(candidate_features_union)
    return candidate_features_union["union"], avg_features


def get_XofN_rules_vouk(candidate_features_for_all_labels):
    candidate_features_unique = deduplicate_XofN_list_groups(candidate_features_for_all_labels)
    candidate_features_union = get_XofN_rules_union(candidate_features_unique)
    avg_features = sum(len(g.split('and')) for g in candidate_features_union["union"]) / len(
        candidate_features_union["union"])
    return candidate_features_union["union"], avg_features


def get_XofN_combs_vouk(candidate_features_for_all_labels):
    candidate_features_unique = deduplicate_full_list_groups(candidate_features_for_all_labels)
    candidate_features_union = get_full_list_union(candidate_features_unique)
    avg_features = print_candidate_features_stats(candidate_features_union)
    return candidate_features_union["union"], avg_features


def deduplicate_XofN_list_groups(candidate_features):
    """
    Deduplicate feature groups per label (string form).
    """
    result = {}
    for label, groups in candidate_features.items():
        # use dict to preserve order
        result[label] = list({g: g for g in groups}.values())
    return result


def get_XofN_rules_union(candidate_features_unique):
    union = set()
    for groups in candidate_features_unique.values():
        for group_str in groups:
            union.add(group_str)
    return {"union": list(union)}


def split_conditions(rule):
    """
    Splits a rule string into atomic conditions.
    """
    return [c.strip() for c in re.split(r"\s+and\s+", rule)]


def add_rule_count_features_vectorized(df, rules, include_original=True, label_columns=None):
    """
    Adds numeric features counting satisfied conditions for each rule.
    Fully vectorized for performance (avoids row-wise loops).
    Labels are always at the end.
    """
    # Sanitize column names for evaluation
    df_sanitized = df.copy()
    df_sanitized.columns = [c.replace(".", "_") for c in df_sanitized.columns]
    rules_sanitized = [r.replace(".", "_") for r in rules]

    # Container for all rule-count series
    rule_count_dict = {}

    for rule in rules_sanitized:
        conditions = split_conditions(rule)
        # Evaluate all conditions into a boolean DataFrame
        condition_df = pd.DataFrame({cond.replace(" ", ""): df_sanitized.eval(cond).astype(int)
                                     for cond in conditions})
        # Sum across conditions ﷿﷿﷿ number of satisfied conditions
        col_name = rule.replace(" ", "").replace(".", "")
        rule_count_dict[col_name] = condition_df.sum(axis=1)

    # Create rule-count DataFrame
    rule_df = pd.DataFrame(rule_count_dict)

    # Combine with original features if requested
    if include_original:
        new_df = pd.concat([df_sanitized, rule_df], axis=1)
    else:
        new_df = rule_df.copy()

    # Always include label columns
    if label_columns is not None:
        for label_col in label_columns:
            new_df[label_col] = df[label_col]  # original labels
        # Move labels to the end
        cols = [c for c in new_df.columns if c not in label_columns] + label_columns
        new_df = new_df[cols]

    return new_df


def get_XofN_feat_groups(candidate_rules_for_all_labels):
    candidate_features_for_all_labels = candidate_rules_for_all_labels.copy()
    for label in candidate_features_for_all_labels:
        rules = candidate_features_for_all_labels[label]
        feature_groups = []
        for rule in rules:
            conditions = rule.split("and")
            XofN_groups = [(condition.replace(" ", "")
                            .replace(">=1", "")
                            .replace("<=1", "")
                            .replace(">=0", "")
                            .replace("<=0", "")
                            .replace("=0", "")
                            .replace("=1", "")
                            .replace("(", "")
                            .replace(")", "")) for condition in conditions]
            if len(conditions) > 1:
                feature_groups.append(XofN_groups)
        candidate_features_for_all_labels[label] = feature_groups
    candidate_features_unique = {
        label: list({tuple(g): g for g in groups}.values())
        for label, groups in candidate_features_for_all_labels.items()
    }
    union = set()
    for groups in candidate_features_unique.values():
        for group in groups:
            union.add(tuple(group))  # convert to tuple for hashing
    candidate_features_union = {"union": [list(group) for group in union]}
    avg_features = print_candidate_features_stats(candidate_features_union)
    return candidate_features_union["union"], avg_features


def get_used_count(org_features, XofN_groups):
    used_features = {feature for group in XofN_groups for feature in group}
    count_used = len(set(org_features) & used_features)
    unused = set(org_features) - used_features
    count_unused = len(unused)
    lengths = [len(group) for group in XofN_groups]
    max_length = max(lengths)
    min_length = min(lengths)
    return count_used, count_unused, max_length, min_length


def get_length_freq(XofN_groups):
    length_freq = {}
    lengths = [len(group) for group in XofN_groups]
    counts = Counter(lengths)
    for length in sorted(counts):
        length_freq[length] = counts[length]
    return length_freq

In [5]:
# tests
df = pd.DataFrame({
    "f_0": [0, 1, 0, 1, 1, 0, 1, 0, 1, 0],
    "f_1": [1, 0, 1, 1, 0, 1, 0, 1, 0, 1],
    "f_2": [0, 1, 1, 0, 1, 0, 0, 1, 1, 0],
    "f_3": [1, 0, 1, 0, 0, 1, 1, 0, 0, 1],
    "f_4": [0, 1, 0, 1, 1, 0, 1, 1, 0, 0],
    "f_5": [0, 1, 0, 1, 1, 0, 1, 1, 0, 0],
    "f_6": [0, 1, 0, 1, 1, 0, 1, 1, 0, 0],
    "f_7": [0, 1, 0, 1, 1, 0, 1, 1, 0, 0],
    "f_8": [0, 1, 0, 1, 1, 0, 1, 1, 0, 0],
    "label": [1, 0, 1, 0, 1, 1, 0, 1, 0, 1]
})
rules = [
    "(f_0 >= 1) and (f_1 >= 1)",
    "(f_1 >= 1) and (f_2 <= 0)",
    "(f_3 >= 1)",
    "(f_3 <= 0) and (f_4 >= 1) and (f_0 >= 1)",
    "(f_1 >= 1) and (f_2 >= 1)",
    "(f_4 >= 1)"
]
result = add_rule_count_features_vectorized(df, rules, True, ["label"])

XofN_groups, avg = get_XofN_feat_groups({"label": rules})
features = ["f_0", "f_1", "f_2", "f_3", "f_4", "f_5", "f_6", "f_7", "f_8"]
used_count = get_used_count(features, XofN_groups)
print(used_count)

result


Label: union
Number of groups: 3
Group length distribution:
  Length 2: 2 groups
  Length 3: 1 groups
Avg length: 2.33, Min length: 2, Max length: 3
(5, 4, 3, 2)


,f_0,f_1,f_2,f_3,f_4,f_5,f_6,f_7,f_8,(f_0>=1)and(f_1>=1),(f_1>=1)and(f_2<=0),(f_3>=1),(f_3<=0)and(f_4>=1)and(f_0>=1),(f_1>=1)and(f_2>=1),(f_4>=1),label
0,0,1,0,1,0,0,0,0,0,1,2,1,0,1,0,1
1,1,0,1,0,1,1,1,1,1,1,0,0,3,1,1,0
2,0,1,1,1,0,0,0,0,0,1,1,1,0,2,0,1
3,1,1,0,0,1,1,1,1,1,2,2,0,3,1,1,0
4,1,0,1,0,1,1,1,1,1,1,0,0,3,1,1,1
5,0,1,0,1,0,0,0,0,0,1,2,1,0,1,0,1
6,1,0,0,1,1,1,1,1,1,1,1,1,2,0,1,0
7,0,1,1,0,1,1,1,1,1,1,1,0,2,2,1,1
8,1,0,1,0,0,0,0,0,0,1,0,0,2,1,0,0
9,0,1,0,1,0,0,0,0,0,1,2,1,0,1,0,1


In [6]:
# tests
candidate_features_unique = {
    "se_C0016382": [
        ["f_347", "f_356"],
        ["f_178", "f_356", "f_380"],
    ],
    "se_C0003811": [["f_347", "f_356"]],
    "se_C0428977": [["f_347", "f_356"], ["f_347", "f_352"]],
    "se_C0027051": [["f_347", "f_356"], ["f_347", "f_355"]],
    "se_C0018790": [["f_347", "f_356"], ["f_347", "f_355"]]
}

intersection = get_full_list_intersection(candidate_features_unique)
print(intersection)

union = get_full_list_union(candidate_features_unique)
print(union)

candidate_features_unique = {
    "se_C0016382": [
        "f_347 and f_356",
        "f_178 and f_356 and f_380",
    ],
    "se_C0003811": ["f_347 and f_356"],
    "se_C0428977": ["f_347 and f_356", "f_347 and f_352"],
    "se_C0027051": ["f_347 and f_356", "f_347 and f_355"],
    "se_C0018790": ["f_347 and f_356", "f_347 and f_355"]
}

union = get_XofN_rules_vouk(candidate_features_unique)
print(union)

candidate_features_unique = {
    "se_C0016382": [
        ["f_347", "f_356"],
        ["f_178", "f_356", "f_380"],
    ],
    "se_C0003811": [["f_347", "f_356"]],
    "se_C0428977": [["f_347", "f_356"], ["f_347", "f_352"]],
    "se_C0027051": [["f_347", "f_356"], ["f_347", "f_355"]],
    "se_C0018790": [["f_347", "f_356"], ["f_347", "f_355"]]
}

union = get_XofN_combs_vouk(candidate_features_unique)
print(union)

{'intersection': [['f_347', 'f_356']]}
{'union': [['f_347', 'f_356'], ['f_178', 'f_356', 'f_380'], ['f_347', 'f_355'], ['f_347', 'f_352']]}
(['f_347 and f_352', 'f_347 and f_355', 'f_347 and f_356', 'f_178 and f_356 and f_380'], 2.25)

Label: union
Number of groups: 4
Group length distribution:
  Length 2: 3 groups
  Length 3: 1 groups
Avg length: 2.25, Min length: 2, Max length: 3
([['f_347', 'f_356'], ['f_178', 'f_356', 'f_380'], ['f_347', 'f_355'], ['f_347', 'f_352']], 2.25)


In [7]:
k = 10
random_state = 42
performances = []
include_original_features_options = [True, False]
training_algorithm = "Variance Reduction"
eval_criteria = ["averageAUROC", "HammingLoss", "SubsetAccuracy", "RankingLoss", "MacroPrecision", "MacroRecall",
                 "MacroFOne"]
max_size = 5
cv_results = []
vouk_path = "C:/Users/Voror/Projects/Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/vouk_folds"

for idx, path in enumerate(paths, start=1):
    run_config = f"\n--- Running with label:'{path["label_set"]}' max_size:'{max_size}' ---"
    print(run_config)
    run_config_name = "_".join(
        [
            path["dataset_name"],
            "_".join(eval_criteria),
            str(max_size),
        ]
    )

    # Load dataset
    current_df = pd.read_csv(path["dataset_path"])
    features = get_features(current_df, path["label_set"])
    # current_df = current_df[features[:10] + path["label_set"]]
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    for fold, (train_idx, test_idx) in enumerate(kf.split(current_df), start=1):
        title = f"\nFold {fold}/{k} ({path["dataset_name"]} {idx}/{len(paths)})"
        print(title)
        # if "Fold 1/10 (CPI+fingerprint_cardiac 1/12)" not in title or fold != 1:
        #     continue
        train_dataset = current_df.iloc[train_idx]
        test_dataset = current_df.iloc[test_idx]
        candidate_features_for_all_labels = {}
        candidate_rules_for_all_labels = {}
        for label in path["label_set"]:
            print(label)
            candidate_features_path = f"{vouk_path}/{path["dataset_name"]}_{label}_trainFold_{fold}.arff-XofNCombs.json"
            with open(candidate_features_path) as f:
                candidate_features_for_label = json.load(f)
                candidate_features_for_all_labels[label] = candidate_features_for_label
            candidate_rules_path = f"{vouk_path}/{path["dataset_name"]}_{label}_trainFold_{fold}.arff-XofNRules.json"
            with open(candidate_rules_path) as f:
                candidate_rules_for_label = json.load(f)
                candidate_rules_for_all_labels[label] = candidate_rules_for_label
            gen_XofN_time_path = f"{vouk_path}/{path["dataset_name"]}_{label}_trainFold_{fold}.arff-time.txt"
            with open(gen_XofN_time_path, "r", encoding="utf-8") as f:
                string_time = f.read().strip()
                if string_time.count('.') > 1:
                    gen_XofN_time = float('.'.join(string_time.split('.', 2)[:2]))
                else:
                    gen_XofN_time = float(string_time)
        XofN_groupings, avg_features, = get_XofN_feat_groups(candidate_rules_for_all_labels)
        count_used, count_unused, max_length, min_length = get_used_count(features, XofN_groupings)
        XofN_rules, avg_rules = get_XofN_rules_vouk(candidate_rules_for_all_labels)
        if len(XofN_rules) == 0:
            print("no XofN groupings were created")
        else:
            for include_original_features in include_original_features_options:
                current_train_dataset = add_rule_count_features_vectorized(
                    train_dataset,
                    XofN_rules,
                    include_original_features,
                    path["label_set"]
                )

                current_test_dataset = add_rule_count_features_vectorized(
                    test_dataset,
                    XofN_rules,
                    include_original_features,
                    path["label_set"]
                )

                current_train_dataset.to_csv(f"vouk/tmp/train_dataset.csv", index=False)
                current_test_dataset.to_csv(f"vouk/tmp/test_dataset.csv", index=False)

                training_start = time.perf_counter()
                original_res, pruned_res, training_time = run_PCT(clus_path,
                                                                  "vouk/tmp/train_dataset.csv",
                                                                  path["label_set"],
                                                                  eval_criteria,
                                                                  test_dataset_path=f"vouk/tmp/test_dataset.csv")
                pruned_performance = get_fold_results(pruned_res, eval_criteria, True, fold, include_original_features,
                                                      XofN_groupings,
                                                      gen_XofN_time,
                                                      training_time, path["dataset_name"])
                pruned_performance["#used"] = count_used
                pruned_performance["#unused"] = count_unused
                pruned_performance["group_max_len"] = max_length
                pruned_performance["group_min_len"] = min_length

                performances.append(pruned_performance)
                performance = get_fold_results(original_res, eval_criteria, False, fold, include_original_features,
                                               XofN_groupings,
                                               gen_XofN_time,
                                               training_time, path["dataset_name"])
                performance["#used"] = count_used
                performance["#unused"] = count_unused
                performance["group_max_len"] = max_length
                performance["group_min_len"] = min_length
                performances.append(performance)

    if len(performances) == 0:
        print("no XofN groupings were created in any fold")
    else:
        final_perf_df = pd.DataFrame(performances)
        averages = final_perf_df.groupby(["pruning", 'include_original_features', 'dataset'])[
            eval_criteria + ['nodes', 'leaves', 'groups',
                             'avg_group_features', 'gen_XofN_time', 'training_time', "#used",
                             "#unused", "group_max_len", "group_min_len"]].mean().reset_index()
        print(averages)
        cv_results.append(averages)
        performances = []
# paths[0] - features[:10] desktop 0.23m
# laptop ??m
# desktop 4h
cv_results


--- Running with label:'['se_C0009676', 'se_C0041657', 'se_C0002994', 'se_C0042571', 'se_C0004604', 'se_C0041834', 'se_C0085631', 'se_C0040822', 'se_C0042373', 'se_C0021053']' max_size:'5' ---

Fold 1/10 (CPI+fingerprint_ten_mid 1/3)
se_C0009676
se_C0041657
se_C0002994
se_C0042571
se_C0004604
se_C0041834
se_C0085631
se_C0040822
se_C0042373
se_C0021053

Label: union
Number of groups: 164
Group length distribution:
  Length 2: 81 groups
  Length 3: 36 groups
  Length 4: 26 groups
  Length 5: 19 groups
  Length 6: 2 groups
Avg length: 2.93, Min length: 2, Max length: 6
pruning: True, include_original_features: with_org, averageAUROC: 0.5584955, HammingLoss: 0.3741, SubsetAccuracy: 0.11511, RankingLoss: 0.35178, MacroPrecision: 0.46395, MacroRecall: 0.27008, MacroFOne: 0.33582, 
pruning: False, include_original_features: with_org, averageAUROC: 0.5934919, HammingLoss: 0.4036, SubsetAccuracy: 0.064748, RankingLoss: 0.33272, MacroPrecision: 0.43794, MacroRecall: 0.46238, MacroFOne: 0.4462, 

[   pruning include_original_features                  dataset  averageAUROC  \
 0    False                    no_org  CPI+fingerprint_ten_mid      0.587852   
 1    False                  with_org  CPI+fingerprint_ten_mid      0.596066   
 2     True                    no_org  CPI+fingerprint_ten_mid      0.597372   
 3     True                  with_org  CPI+fingerprint_ten_mid      0.593472   
 
    HammingLoss  SubsetAccuracy  RankingLoss  MacroPrecision  MacroRecall  \
 0     0.403220        0.049062     0.343826        0.429449     0.504815   
 1     0.400643        0.046883     0.341873        0.432337     0.503252   
 2     0.338627        0.124861     0.344552        0.521514     0.278628   
 3     0.340580        0.127019     0.349641        0.515029     0.270106   
 
    MacroFOne   nodes  leaves  groups  avg_group_features  gen_XofN_time  \
 0   0.461511  1017.6   509.3   176.2             2.86697      19.444921   
 1   0.461660  1015.2   508.1   176.2             2.86697  

In [8]:
# All results
save_path = "vouk/"
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res.sort_values(by='dataset', ascending=False, inplace=True)
final_grouped_res = final_grouped_res.reset_index(drop=True)
final_grouped_res.to_csv(save_path + "all_results.csv")
final_grouped_res

,pruning,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time,#used,#unused,group_max_len,group_min_len
0,False,no_org,fingerprint_ten_mid,0.577190,0.416483,0.041175,0.344481,0.422841,0.518177,0.461898,974.6,487.8,152.9,2.923478,4.471284,0.812904,131.9,408.1,7.3,2.0
1,False,with_org,fingerprint_ten_mid,0.573790,0.418851,0.038914,0.346224,0.417535,0.498259,0.450480,984.4,492.7,152.9,2.923478,4.471284,1.427772,131.9,408.1,7.3,2.0
2,True,no_org,fingerprint_ten_mid,0.575116,0.359186,0.115966,0.351840,0.473041,0.234398,0.307942,80.4,40.7,152.9,2.923478,4.471284,0.812904,131.9,408.1,7.3,2.0
3,True,with_org,fingerprint_ten_mid,0.570466,0.369263,0.110618,0.353624,0.448874,0.251586,0.314417,102.8,51.9,152.9,2.923478,4.471284,1.427772,131.9,408.1,7.3,2.0
4,False,no_org,CPI_ten_mid,0.573231,0.406507,0.058706,0.351524,0.442990,0.435887,0.436201,627.8,314.4,292.6,2.450513,14.848584,1.018486,125.7,1481.3,6.6,2.0
5,False,with_org,CPI_ten_mid,0.572139,0.405571,0.047830,0.346000,0.446567,0.467511,0.454298,767.8,384.4,292.6,2.450513,14.848584,2.953684,125.7,1481.3,6.6,2.0
6,True,no_org,CPI_ten_mid,0.570984,0.354561,0.112071,0.374618,0.523884,0.265687,0.345613,57.6,29.3,292.6,2.450513,14.848584,1.018486,125.7,1481.3,6.6,2.0
7,True,with_org,CPI_ten_mid,0.577320,0.358534,0.109352,0.365202,0.516205,0.288532,0.361943,75.2,38.1,292.6,2.450513,14.848584,2.953684,125.7,1481.3,6.6,2.0
8,False,no_org,CPI+fingerprint_ten_mid,0.587852,0.403220,0.049062,0.343826,0.429449,0.504815,0.461511,1017.6,509.3,176.2,2.866970,19.444921,0.907588,149.8,1997.2,6.1,2.0
9,False,with_org,CPI+fingerprint_ten_mid,0.596066,0.400643,0.046883,0.341873,0.432337,0.503252,0.461660,1015.2,508.1,176.2,2.866970,19.444921,3.404018,149.8,1997.2,6.1,2.0


In [9]:
# Table ready (with pruning, rounded, compact)
save_path = "vouk/"
rounded_final_grouped_res = pd.read_csv(save_path + "all_results.csv", index_col=0)
res = pd.read_csv(save_path + "all_results.csv", index_col=0)
table_results = get_table_results(res, eval_criteria, get_dataset_paths())
table_results.to_csv(save_path + "table_results.csv")
table_results

,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,#used,#unused,group_max_len,group_min_len,Nodes; Leaves,#XofN; #Feat/XofN,# Ung. Feats,XofN time (s); PCT tr. time (s)
2,no_org,fingerprint_ten_mid,0.575,0.359,0.116,0.352,0.473,0.234,0.308,131.9,408.1,7.3,2.0,80.4; 40.7,152.9; 2.9,408.1,4.5; 0.8
3,with_org,fingerprint_ten_mid,0.570,0.369,0.111,0.354,0.449,0.252,0.314,131.9,408.1,7.3,2.0,102.8; 51.9,152.9; 2.9,408.1,4.5; 1.4
6,no_org,CPI_ten_mid,0.571,0.355,0.112,0.375,0.524,0.266,0.346,125.7,1481.3,6.6,2.0,57.6; 29.3,292.6; 2.5,1481.3,14.8; 1.0
7,with_org,CPI_ten_mid,0.577,0.359,0.109,0.365,0.516,0.289,0.362,125.7,1481.3,6.6,2.0,75.2; 38.1,292.6; 2.5,1481.3,14.8; 3.0
10,no_org,CPI+fingerprint_ten_mid,0.597,0.339,0.125,0.345,0.522,0.279,0.357,149.8,1997.2,6.1,2.0,93.2; 47.1,176.2; 2.9,1997.2,19.4; 0.9
11,with_org,CPI+fingerprint_ten_mid,0.593,0.341,0.127,0.350,0.515,0.270,0.349,149.8,1997.2,6.1,2.0,106.6; 53.8,176.2; 2.9,1997.2,19.4; 3.4


In [ ]:
k = 10
random_state = 42
vouk_path = "C:/Users/Voror/Projects/Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/vouk_folds"
length_frequency = {}
for idx, path in enumerate(paths, start=1):
    run_config = f"\n--- Running with label:'{path["label_set"]}' ---"
    print(run_config)
    run_config_name = path["dataset_name"]

    # Load dataset
    current_df = pd.read_csv(path["dataset_path"])
    features = get_features(current_df, path["label_set"])
    # current_df = current_df[features[:10] + path["label_set"]]
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    for fold, (train_idx, test_idx) in enumerate(kf.split(current_df), start=1):
        title = f"\nFold {fold}/{k} ({path["dataset_name"]} {idx}/{len(paths)})"
        print(title)
        train_dataset = current_df.iloc[train_idx]
        test_dataset = current_df.iloc[test_idx]
        candidate_features_for_all_labels = {}
        candidate_rules_for_all_labels = {}
        for label in path["label_set"]:
            candidate_features_path = f"{vouk_path}/{path["dataset_name"]}_{label}_trainFold_{fold}.arff-XofNCombs.json"
            with open(candidate_features_path) as f:
                candidate_features_for_label = json.load(f)
                candidate_features_for_all_labels[label] = candidate_features_for_label
            candidate_rules_path = f"{vouk_path}/{path["dataset_name"]}_{label}_trainFold_{fold}.arff-XofNRules.json"
            with open(candidate_rules_path) as f:
                candidate_rules_for_label = json.load(f)
                candidate_rules_for_all_labels[label] = candidate_rules_for_label
            gen_XofN_time_path = f"{vouk_path}/{path["dataset_name"]}_{label}_trainFold_{fold}.arff-time.txt"
            with open(gen_XofN_time_path, "r", encoding="utf-8") as f:
                gen_XofN_time = float(f.read().strip())
        XofN_groupings, avg_features, = get_XofN_feat_groups(candidate_rules_for_all_labels)
        count_used, count_unused, max_length, min_length = get_used_count(features, XofN_groupings)
        histogram = get_length_freq(XofN_groupings)
        print(histogram)
        if path["dataset_name"] not in length_frequency:
            length_frequency[path["dataset_name"]] = {}
        length_frequency[path["dataset_name"]][fold] = histogram
        XofN_rules, avg_rules = get_XofN_rules_vouk(candidate_rules_for_all_labels)

length_frequency

In [ ]:
import pandas as pd

rows = []

for dataset, folds in length_frequency.items():
    for fold, lengths in folds.items():
        for length, freq in lengths.items():
            rows.append((dataset, fold, length, freq))

df = pd.DataFrame(rows, columns=["dataset", "fold", "length", "freq"])

avg_df = df.groupby(["dataset", "length"])["freq"].mean().reset_index()
avg_df["avg_freq"] = avg_df["freq"].round(2)
avg_df.drop("freq", axis=1, inplace=True)
avg_df.to_csv("length_frequency_histogram.csv", index=False)